In [ ]:
!pip install --upgrade transformers datasets huggingface-hub peft evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 30.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [ ]:
# lora_vs_adapter.py
# Compare LoRA vs Adapter tuning on a downstream classification task (SST-2 subset).
# Every line contains a detailed inline comment explaining what it does and why.

# Install the evaluate library if it's not already installed
!pip install -q evaluate datasets huggingface-hub --upgrade

# -------------------------
# 0) Imports and environment setup
# -------------------------
import os                                                    # for environment / file ops if needed
import random                                                # random utilities used for deterministic subsampling
import numpy as np                                           # numpy for deterministic seeding & numeric ops
from dataclasses import dataclass                             # dataclass for clean config objects
from typing import Dict, Any                                 # typing helpers for function signatures

import torch                                                  # PyTorch core (tensors, device, etc.)
import torch.nn as nn                                         # neural network layers (we'll build an adapter)
from torch.utils.data import DataLoader                       # DataLoader for batching custom DataSets if needed

from datasets import load_dataset                             # Hugging Face datasets to load SST-2
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# optional evaluation library to compute accuracy
import evaluate                                               # HF evaluate for accuracy metric

# PEFT provides a convenient LoRA wrapper
from peft import LoraConfig, get_peft_model  # LoRA utilities from PEFT

# -------------------------
# 1) Quick config
# -------------------------
@dataclass
class Config:
    """Small configuration container to keep hyperparameters organized."""
    model_name: str = "distilbert-base-uncased"              # base model to use (small & fast)
    dataset_name: str = "glue"                               # dataset hub name (SST-2 lives under 'glue')
    dataset_config: str = "sst2"                             # specific config name in the dataset hub
    num_train_samples: int = 2000                             # small subset size for quick runs (increase for better estimates)
    num_val_samples: int = 500                                # validation subset size
    batch_size: int = 32                                      # batch size for training & eval
    num_epochs: int = 3                                     # number of epochs (small for demo)
    lr: float = 3e-4                                          # learning rate for adapter / LoRA tuning
    seed: int = 42                                            # random seed for reproducibility
    output_dir: str = "./out_lora_adapter"                    # directory to save trained models & logs

cfg = Config()                                                # instantiate config with defaults

# -------------------------
# 2) Reproducibility: set seeds
# -------------------------
random.seed(cfg.seed)                                        # seed python's RNG
np.random.seed(cfg.seed)                                     # seed numpy RNG
torch.manual_seed(cfg.seed)                                  # seed PyTorch RNG (CPU)
if torch.cuda.is_available():                                 # if GPU is available, seed CUDA RNGs too
    torch.cuda.manual_seed_all(cfg.seed)

# -------------------------
# 3) Load dataset & tokenizer
# -------------------------
# load glue/sst2 dataset (train/validation splits)
#raw_datasets = load_dataset(cfg.dataset_name, cfg.dataset_config, trust_remote_code=True)
raw_datasets = load_dataset("nyu-mll/glue", "sst2")
# create tokenizer from model name (handles casing, vocab, special tokens)
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)

# helper function to preprocess (tokenize) text examples
def preprocess_function(examples):
    """Tokenize examples and return the tokenized batch with default truncation."""
    # the dataset's text column for SST-2 is 'sentence'; tokenizer will produce input_ids + attention_mask
    return tokenizer(examples["sentence"], truncation=True)

# tokenize full datasets with map (this operates lazily and efficiently)
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=["sentence", "idx"] # Remove original text and index columns
)

# -------------------------
# 4) Take small subsets for fast experimentation
# -------------------------
# to keep runtime reasonable in this demo we use small subsets; remove slicing for full experiments
train_dataset = tokenized_datasets["train"].shuffle(seed=cfg.seed).select(range(cfg.num_train_samples))
eval_dataset = tokenized_datasets["validation"].shuffle(seed=cfg.seed).select(range(cfg.num_val_samples))

# collator that pads to longest sample in batch (avoid padding to model max length for speed)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# choose metric object for evaluation
accuracy_metric = evaluate.load("accuracy")

# ---------
# 5) Utility: compute metrics for HF Trainer
# ---------
def compute_metrics(eval_pred):
    """Compute metrics from predictions (HF Trainer expects logits & label_ids)."""
    logits, labels = eval_pred                               # HF passes a tuple (preds, labels)
    preds = np.argmax(logits, axis=-1)                       # pick class with highest logit
    acc = accuracy_metric.compute(predictions=preds, references=labels)  # compute accuracy
    return {"accuracy": acc["accuracy"]}

# -------------------------
# 6) Baseline factory: base model loader & freeze helper
# -------------------------
def get_base_model_and_tokenizer():
    """Load the pretrained model for sequence classification and return it (fresh copy)."""
    # AutoModelForSequenceClassification gives a model with a classification head suitable for glue tasks
    model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name, num_labels=2)
    return model

def freeze_model_params(model):
    """Freeze all parameters of the given model (used when training small adapters)."""
    for param in model.parameters():                         # iterate over all parameters in the module
        param.requires_grad = False                          # set requires_grad False to freeze parameter during training

# -------------------------
# 7) Implementation A — LoRA tuning using PEFT
# -------------------------
def train_with_lora():
    """Configure and train a LoRA-adapted model using PEFT and HuggingFace Trainer."""
    # 1) load a fresh base model for classification
    model = get_base_model_and_tokenizer()

    # 2) prepare model for int8/training if we wanted quantization - not used here but safe call if required by PEFT examples
    #    we won't quantize in this demo, so we skip prepare_model_for_int8_training; leave model as-is.

    # 3) Define LoRA configuration — rank, alpha, and target modules
    lora_config = LoraConfig(
        r=8,                                                  # rank of LoRA decomposition (small -> fewer parameters)
        lora_alpha=32,                                        # scaling factor for LoRA
        target_modules=["q_lin", "k_lin", "v_lin", "out_lin"], # target DistilBERT attention layers for LoRA injection
        lora_dropout=0.1,                                     # dropout applied to LoRA path
        bias="none",                                          # whether to adapt bias (none here)
        task_type="SEQ_CLS"                                   # task type (sequence classification)
    )

    # 4) wrap the model with PEFT to add LoRA adapters (this returns a model-like object where only LoRA params are trainable)
    model = get_peft_model(model, lora_config)

    # 5) print how many parameters are trainable (sanity check)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"[LoRA] Trainable params: {trainable_params} / {total_params} ({100*trainable_params/total_params:.2f}%) Kishan)")

    # 6) training arguments for HF Trainer
    training_args = TrainingArguments(
        output_dir=os.path.join(cfg.output_dir, "lora"),      # save outputs to distinct subdirectory
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        learning_rate=cfg.lr,
        num_train_epochs=cfg.num_epochs,
        eval_strategy="epoch",                          # evaluate after each epoch
        save_strategy="epoch",                                # checkpoint each epoch
        logging_strategy="epoch",
        remove_unused_columns=False,                          # keep all columns (our collator handles padding)
        seed=cfg.seed,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none" # Disable wandb logging
    )

    # 7) instantiate HF Trainer; pass the model (LoRA-wrapped), datasets, collator, and metric function
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        #tokenizer=tokenizer,  # Removed as data_collator handles it
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    # 8) train the LoRA-adapted model
    trainer.train()

    # 9) evaluate final model on eval split
    eval_res = trainer.evaluate(eval_dataset=eval_dataset)
    print(f"[LoRA] Eval results: {eval_res}")

    # 10) return the trainer and evaluation results for comparison later
    return trainer, eval_res

# -------------------------
# 8) Implementation B — Adapter tuning (simple bottleneck adapter)
# -------------------------
class AdapterHead(nn.Module):
    """A compact bottleneck adapter added to the pooled output of the base model.
    This adapter is intentionally simple: down-project -> nonlinearity -> up-project.
    During adapter training we freeze the backbone and train only this module + classifier head.
    """
    def __init__(self, hidden_size: int, adapter_size: int = 64):
        super().__init__()                                     # initialize nn.Module internals
        self.down_proj = nn.Linear(hidden_size, adapter_size)  # projection to low-dim bottleneck
        self.activation = nn.ReLU()                            # nonlinearity inside adapter
        self.up_proj = nn.Linear(adapter_size, hidden_size)    # project back to original hidden size

    def forward(self, x):
        """Forward pass for adapter. x shape: (batch, hidden_size). Returns same shape."""
        z = self.down_proj(x)                                  # reduce dimensionality -> (batch, adapter_size)
        z = self.activation(z)                                 # nonlinear transform -> (batch, adapter_size)
        z = self.up_proj(z)                                    # project back -> (batch, hidden_size)
        return z                                               # return residual to be added to original representation

class ModelWithAdapter(nn.Module):
    """Wrap a pretrained classification model; freeze its backbone and add an adapter + small classifier.
    We reuse the model's tokenizer and architecture but replace the classification head to ensure only adapter+head train.
    """
    def __init__(self, base_model_name: str, adapter_size: int = 64, num_labels: int = 2):
        super().__init__()                                     # initialize module
        # load a pretrained classification model which includes a backbone + classifier head
        self.backbone = AutoModelForSequenceClassification.from_pretrained(base_model_name, num_labels=num_labels)
        # freeze all backbone parameters so training only updates adapter + new head
        freeze_model_params(self.backbone)
        # backbone returns logits via its .classifier and also provides the transformer outputs; we will intercept pooled output
        # create adapter module operating on backbone.config.dim (hidden size)
        hidden_size = self.backbone.config.dim                  # DistilBERT's hidden dimension
        self.adapter = AdapterHead(hidden_size, adapter_size)   # instantiate adapter
        # build a fresh classification head that consumes (backbone_pooled + adapter_output) combined representation
        self.classifier = nn.Linear(hidden_size, num_labels)    # simple linear classifier mapping to label logits

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs): # Added **kwargs here
        """Forward method that calls backbone to get pooled output, applies adapter, and classifies."""
        # get outputs from backbone; distilbert returns 'last_hidden_state' (no pooler) -> we compute mean pooling
        outputs = self.backbone.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state                  # shape: (batch, seq_len, hidden_size)
        # compute mean pooling over non-padded tokens: sum followed by dividing by valid counts
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden)  # expand mask to match hidden dims -> (batch, seq_len, 1)
        summed = (last_hidden * mask).sum(dim=1)                 # sum only valid positions -> (batch, hidden)
        counts = mask.sum(dim=1).clamp(min=1e-9)                 # count non-pad tokens per example -> (batch, 1)
        pooled = summed / counts                                 # mean pooled representation -> (batch, hidden)
        # pass pooled vector through adapter and add residually
        adapter_out = self.adapter(pooled)                       # (batch, hidden)
        combined = pooled + adapter_out                          # residual addition -> (batch, hidden)
        # classifier consumes combined representation to produce logits
        logits = self.classifier(combined)                       # (batch, num_labels)
        loss = None
        # if labels provided compute classification loss using CrossEntropy
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            return {"loss": loss, "logits": logits}
        # otherwise just return logits to HF Trainer-like loop
        return {"logits": logits}

def train_with_adapter():
    """Train the model that contains a small adapter inserted at pooled-output stage."""
    # 1) instantiate model with adapter (fresh)
    adapter_model = ModelWithAdapter(cfg.model_name, adapter_size=64, num_labels=2)

    # 2) check parameter counts: adapter + classifier should be a tiny fraction
    trainable_params = sum(p.numel() for p in adapter_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in adapter_model.parameters())
    print(f"[Adapter] Trainable params: {trainable_params} / {total_params} ({100*trainable_params/total_params:.2f}%) Kishan)")

    # 3) We will train using a simple Trainer-like loop by wrapping adapter_model into a HF-compatible interface.
    #    The HuggingFace Trainer expects the model to return (loss, logits) or a ModelOutput object; to simplify we define
    #    a small wrapper using `transformers.Trainer` by implementing a custom model that behaves similar to HF models.
    #    To avoid subclassing HF internals, we'll create a small torch.nn.Module wrapper exposing `forward` signature compatible with Trainer.

    class HFAdapterModule(nn.Module):
        """A thin wrapper to make ModelWithAdapter compatible with HF Trainer's forward signature."""
        def __init__(self, wrapped):
            super().__init__()
            self.wrapped = wrapped

        def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs): # Added **kwargs here
            out = self.wrapped(input_ids=input_ids, attention_mask=attention_mask, labels=labels, **kwargs)
            # if loss present return tuple (loss, logits) compatible with Trainer
            if "loss" in out:
                return (out["loss"], out["logits"])
            return out["logits"]

    wrapped_model = HFAdapterModule(adapter_model)            # create wrapper for use with Trainer

    # 4) training arguments
    training_args = TrainingArguments(
        output_dir=os.path.join(cfg.output_dir, "adapter"),   # separate output directory
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        learning_rate=cfg.lr,
        num_train_epochs=cfg.num_epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        remove_unused_columns=False,
        seed=cfg.seed,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none" # Disable wandb logging
    )

    # 5) However, HF Trainer will try to call `.parameters()` and update weights; we must ensure optimizer updates only adapter+classifier.
    #    By default Trainer will include all parameters, so we provide `optimizers` or set `model.requires_grad` appropriately.
    #    Our adapter_model has frozen backbone parameters via freeze_model_params() earlier; thus only adapter+classifier have requires_grad=True.
    #    That is sufficient for Trainer to update only those parameters.

    # 6) But Trainer expects the model object to be a transformers.PreTrainedModel for some built-in behaviors (save_pretrained etc).
    #    To keep the demo simple, we'll rely on Trainer's basic forward/optimization loop without advanced model checkpointing of HF `PreTrainedModel`.
    #    (This is fine for demonstration; for production you'd subclass PreTrainedModel or integrate adapters into model internals.)

    # 7) prepare a custom compute_metrics that extracts logits when Trainer wraps them
    def compute_metrics_adapter(eval_pred):
        """Adapter wrapper compute_metrics — eval_pred may be (preds, labels) or trainer wraps tuples."""
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        acc = accuracy_metric.compute(predictions=preds, references=labels)
        return {"accuracy": acc["accuracy"]}

    # 8) instantiate Trainer with wrapped model and train
    trainer = Trainer(
        model=wrapped_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        #tokenizer=tokenizer,  # Removed as data_collator handles it
        data_collator=data_collator,
        compute_metrics=compute_metrics_adapter
    )

    # 9) train the adapter (only adapter+classifier parameters have requires_grad=True)
    trainer.train()

    # 10) evaluate on validation data
    eval_res = trainer.evaluate(eval_dataset=eval_dataset)
    print(f"[Adapter] Eval results: {eval_res}")

    # 11) return trainer and results
    return trainer, eval_res

# -------------------------
# 9) Run both experiments and compare
# -------------------------
def main():
    """Train LoRA and Adapter variants and print a compact comparison summary."""
    os.makedirs(cfg.output_dir, exist_ok=True)               # ensure output directory exists

    # 1) Train LoRA
    print("== Starting LoRA training ==")
    lora_trainer, lora_res = train_with_lora()               # train + eval for LoRA

    # 2) Train Adapter
    print("\n== Starting Adapter training ==")
    adapter_trainer, adapter_res = train_with_adapter()      # train + eval for Adapter

    # 3) Compact comparison printout
    print("\n=== Comparison Summary ===")
    print(f"LoRA eval accuracy: {lora_res.get('eval_accuracy', lora_res.get('accuracy', 'N/A'))}")
    print(f"Adapter eval accuracy: {adapter_res.get('eval_accuracy', adapter_res.get('accuracy', 'N/A'))}")
    print("\nNote: These experiments used small dataset subsets and few epochs for speed —\n" \
          "increase dataset size and epochs for reliable comparison in production.")

# -------------------------
# 10) Entrypoint
# -------------------------
if __name__ == "__main__":
    main()

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

== Starting LoRA training ==


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[Adapter] Trainable params: 100674 / 67055684 (0.15%) Kishan)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TypeError: train_with_adapter.<locals>.HFAdapterModule.forward() got an unexpected keyword argument 'token_type_ids'